<div align="center">
<h1 style="color:rgb(121, 65, 6);">Speech Emotion Recognition</h1>
<h3>Modelling</h3>
<a href="">Give Feedback</a> | <a href="">Bug report</a>

</div>

**Tags:** #ser #audio #modelling #classification

**Author:** [Ndeye Awa Salane](https://www.linkedin.com/in/ndeye-awa-salane-a93667230/)

**Last update:** 2025-04-18 (Created: 2025-04-16)

**Description:** This notebook loads the processed (unscaled) feature dataset, performs a train/test split, scales features **after** the split (to prevent data leakage), and trains a classifier.

## Table of Contents

1. [Import libraries](#import-libraries)
2. [Load & prepare data](#load-data)
3. [MLflow setup](#mlflow-setup)
4. [Bayesian hyperparameter sweep](#bayesian-hyperparameter-sweep)
5. [Sweep results analysis](#sweep-results)
6. [Voting Classifier (Ensemble)](#voting-classifier-ensemble)
7. [Best model evaluation](#best-model-evaluation)
8. [Conclusions](#conclusions)
9. [Resources](#resources)

# Import libraries

In [ ]:
import sys, importlib
sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))

# Force-reload src modules so changes take effect without kernel restart
import src.config, src.training, src.sweep, src.evaluate
for mod in [src.config, src.training, src.sweep, src.evaluate]:
    importlib.reload(mod)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from src.config import DATA_PROCESSED, MODELS_DIR
from src.training import prepare_data
from src.sweep import (
    run_sweep, get_best_run, build_voting_classifier, MODEL_REGISTRY,
)
from src.evaluate import (
    plot_confusion_matrix, print_classification_report,
    plot_sweep_results, load_best_model,
)

sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['labels'].value_counts()}")
df.head()

In [ ]:
# Split FIRST, then scale — scaler is fit on train only (no data leakage!)
X_train, X_test, y_train, y_test, scaler = prepare_data(df)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"y_train: {y_train.nunique()} classes  y_test: {y_test.nunique()} classes")

# MLflow setup

We use MLflow to track every training run. Each hyperparameter combination gets its own run with:
- **Parameters** logged (e.g. `n_estimators`, `C`, `kernel`)
- **Metrics** logged (`accuracy`, `f1_weighted`, `precision_weighted`, `recall_weighted`)
- **Model artifact** saved (loadable later with `mlflow.sklearn.load_model`)

Launch the dashboard with: `mlflow ui` (in the project root) → open http://127.0.0.1:5000

In [ ]:
# Point MLflow to store runs inside the project (not in your home folder)
mlflow.set_tracking_uri(str(MODELS_DIR / "mlruns"))

# Show available model types
print("Models in the Bayesian sweep:")
for name in MODEL_REGISTRY:
    print(f"  • {name}")

# Bayesian Hyperparameter Sweep (Optuna TPE)

Instead of brute-force grid search, we use **Optuna's Tree-structured Parzen Estimator (TPE)** to intelligently explore the hyperparameter space. Each model family gets `n_trials_per_model` Bayesian trials, and every trial is logged to MLflow.

In [ ]:
%%time

# Bayesian sweep — 30 TPE trials per model family (150 total)
results_df = run_sweep(
    X_train, X_test, y_train, y_test,
    model_names=None,                        # all 5 model families
    experiment_name="Bayesian_sweep_all_features",
    n_trials_per_model=5,                   # 5 Bayesian trials each
)

print(f"\n Bayesian sweep complete — {len(results_df)} total runs logged to MLflow")
results_df.sort_values("f1_weighted", ascending=False).head(10)

# Sweep results analysis

Compare all runs at a glance. You can also open the MLflow dashboard for interactive exploration:

```bash
cd /Users/ndeyeawasalane/Downloads/SER
mlflow ui --backend-store-uri models/mlruns
```
Then open http://127.0.0.1:5000

In [ ]:
# Visual comparison of all runs
fig = plot_sweep_results(results_df, metric="f1_weighted")
plt.show()

# Top 10 across all model types
print("\nTop 10 runs by F1 (weighted):")
results_df.sort_values("f1_weighted", ascending=False)[
    ["run_name", "model_type", "accuracy", "f1_weighted"]
].head(10)

# Voting Classifier (Ensemble)

Build a **Soft-Voting Classifier** from the top-3 best model families found in the sweep. Soft voting averages the predicted probabilities, which typically outperforms hard (majority) voting.

In [ ]:
# Build a Voting Classifier from the top 3 model families
voting_model, voting_metrics, voting_run_id = build_voting_classifier(
    results_df,
    X_train, X_test, y_train, y_test,
    top_n=3,
    voting="soft",
    experiment_name="Bayesian_sweep_all_features",
)

# Add the voting classifier to the results for comparison
voting_row = pd.DataFrame([{
    "run_id": voting_run_id,
    "model_type": "VotingClassifier",
    "run_name": "VotingClassifier_soft",
    **voting_metrics,
}])
results_df = pd.concat([results_df, voting_row], ignore_index=True)

print(f"\nUpdated leaderboard (with VotingClassifier):")
results_df.sort_values("f1_weighted", ascending=False)[
    ["run_name", "model_type", "accuracy", "f1_weighted"]
].head(10)

# Best model evaluation

Load the overall best model (individual or VotingClassifier) and do a detailed evaluation: confusion matrix and per-class classification report.

In [ ]:
# Identify and reload the best model
best = get_best_run(results_df, metric="f1_weighted")
print(f"🏆 Best run: {best['run_name']}  ({best['model_type']})")
print(f"   Accuracy:  {best['accuracy']:.4f}")
print(f"   F1:        {best['f1_weighted']:.4f}")
print(f"   Precision: {best['precision_weighted']:.4f}")
print(f"   Recall:    {best['recall_weighted']:.4f}")

best_model = load_best_model(best["run_id"])

In [ ]:
# Classification report
y_pred = best_model.predict(X_test)
print_classification_report(y_test, y_pred)

In [ ]:
# Confusion matrix
fig = plot_confusion_matrix(y_test, y_pred,
                            title=f"Confusion Matrix — {best['model_type']}")
plt.show()

In [ ]:
# Save the best model locally + log the confusion matrix figure to the best run
import joblib

# Save as a standalone joblib file
joblib.dump(best_model, MODELS_DIR / "best_model.joblib")

# Log confusion matrix as an artifact on the best run
with mlflow.start_run(run_id=best["run_id"]):
    fig.savefig(MODELS_DIR / "confusion_matrix.png", dpi=150)
    mlflow.log_artifact(str(MODELS_DIR / "confusion_matrix.png"))

print(f"Best model saved to models/best_model.joblib")
print(f"Confusion matrix logged to MLflow run {best['run_id'][:8]}...")

# Conclusions

We ran a **Bayesian hyperparameter sweep** (Optuna TPE) across 5 model families (Random Forest, SVM, Gradient Boosting, KNN, Logistic Regression) with 30 trials each, plus a **Soft-Voting Classifier** ensemble built from the top-3 families. All results are tracked in MLflow.

```bash
cd /Users/ndeyeawasalane/Downloads/SER
mlflow ui --backend-store-uri models/mlruns
```

# Resources

- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
- [MLflow Tracking Guide](https://mlflow.org/docs/latest/tracking.html)
- [scikit-learn Model Selection](https://scikit-learn.org/stable/model_selection.html)
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Audio Features](https://ravinkumar.com/GenAiGuidebook/audio/audio_feature_extraction.html)